# Funding-Rate Carry Backtest with `candlefeed`

Perpetual funding is a recurring cash flow: when funding is **positive**, shorts get paid by longs. A classic market-neutral carry trade is to **short the perp and hold spot** (or hedge on another venue), collecting funding while staying delta-flat.

This notebook uses the official [CandleFeed](https://candlefeed.ai) Python client to:

1. Pull **OI-weighted aggregated funding** across Binance / Bybit / OKX (free-tier-friendly).
2. Visualise the carry and its cumulative payoff.
3. Run a simple **short-perp carry backtest** on the full-history per-exchange funding series (a paid-tier range) and compute annualised yield.

> Every call returns a tidy pandas DataFrame indexed by time — so the whole thing is `.cumsum()` and `.plot()` from here.

In [ ]:
# pip install candlefeed matplotlib
import os
import pandas as pd
import matplotlib.pyplot as plt
from candlefeed import CandleFeed

# Set CANDLEFEED_API_KEY in your environment, or pass api_key="cf_live_..."
cf = CandleFeed(api_key=os.environ.get("CANDLEFEED_API_KEY"))
cf

## 1. Aggregated funding across venues (free tier)

`get_funding_rates_aggregated` returns the open-interest-weighted funding rate across the exchanges where CandleFeed has both funding and OI — the single number that best represents "the market's" funding for a symbol.

In [ ]:
agg = cf.get_funding_rates_aggregated(
    "BTCUSDT",
    interval="1h",
    exchanges=["binance", "bybit", "okx"],
    start="2024-01-01",
    end="2024-04-01",
)
agg[["weighted_funding_rate", "total_oi_usd", "exchange_count"]].head()

In [ ]:
# A short-perp carry earns the funding rate each period when funding > 0.
# Cumulative carry = running sum of the weighted funding rate.
carry = agg["weighted_funding_rate"].astype(float)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
carry.plot(ax=ax1, lw=0.8)
ax1.axhline(0, color="k", lw=0.6)
ax1.set_title("BTCUSDT OI-weighted funding rate (per interval)")
ax1.set_ylabel("funding rate")

carry.cumsum().plot(ax=ax2, color="C2")
ax2.set_title("Cumulative short-perp carry (sum of funding collected)")
ax2.set_ylabel("cumulative funding")
plt.tight_layout()

## 2. Per-exchange funding — which venue pays best?

Funding isn't uniform across exchanges. Pulling each venue's raw funding series lets you pick the richest one to short. `get_funding_rates` auto-paginates the full requested range into one frame.

In [ ]:
venues = ["binance", "bybit", "okx"]
series = {}
for ex in venues:
    fr = cf.get_funding_rates("BTCUSDT", exchange=ex, start="2024-01-01", end="2024-04-01")
    series[ex] = fr["funding_rate"].astype(float)

funding = pd.DataFrame(series).sort_index()
funding.describe()

In [ ]:
# Resample to daily realised funding per venue and compare cumulative payoff.
daily = funding.resample("1D").sum()
daily.cumsum().plot(figsize=(11, 5), title="Cumulative funding collected by shorting the perp, by venue")
plt.ylabel("cumulative funding")
plt.axhline(0, color="k", lw=0.6);

## 3. Carry backtest + annualised yield (paid-tier full history)

Below we extend the window to **full history** on the richest venue. Free tier is limited to the last 30 days — the cell below needs **Builder or above** (it will raise `TierRestrictedError` on free, which we catch and explain).

The strategy: short 1 unit of perp, hold the hedge, collect funding every interval. We approximate the gross carry PnL as the cumulative funding and annualise it.

In [ ]:
from candlefeed import TierRestrictedError

BEST_VENUE = "binance"  # swap for whichever venue paid best above

try:
    fr = cf.get_funding_rates("BTCUSDT", exchange=BEST_VENUE, start="2021-01-01", end="2024-12-31")
    rate = fr["funding_rate"].astype(float)

    # Funding posts every 8h on Binance => 3 intervals/day, ~1095/yr.
    periods_per_year = 365 * 3
    mean_per_period = rate.mean()
    annualised = mean_per_period * periods_per_year

    print(f"Periods:                 {len(rate):,}")
    print(f"Mean funding / period:   {mean_per_period:.6%}")
    print(f"Total funding collected: {rate.sum():.4%}")
    print(f"Annualised carry yield:  {annualised:.2%}")

    rate.cumsum().plot(figsize=(11, 5), color="C3",
                       title=f"BTCUSDT short-perp cumulative carry — {BEST_VENUE}, full history")
    plt.ylabel("cumulative funding")
    plt.axhline(0, color="k", lw=0.6)
except TierRestrictedError as e:
    print("This deep-history backtest needs a paid plan:")
    print(" ", e.message)

## Caveats

This is a teaching example, not a production strategy. A real carry book must account for:

- **Hedge cost & basis drift** — shorting the perp leaves you long spot (or long another venue); financing and basis moves both bite.
- **Funding sign flips** — funding goes negative in bearish regimes, inverting the trade.
- **Fees, slippage, and liquidation risk** on the short leg.

Layer in OHLCV (`cf.get_ohlcv`), open interest (`cf.get_open_interest`), and basis (`cf.get_basis`) — or pull them time-aligned in one call with `cf.get_combined(..., fields=["ohlcv", "funding_rate", "open_interest"])` — to build the full picture.

Docs and API keys: **[candlefeed.ai](https://candlefeed.ai)**.